In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day05" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w3" / "day05"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

프로젝트 루트  : c:\hw\hw\python_basic\hanwha-agent


In [2]:
from langchain_anthropic import ChatAnthropic

# claude chat model 생성
llm= ChatAnthropic(
    model="claude-haiku-4-5",
    max_tokens=500,
)

response = llm.invoke(
    "RAG가 무엇인지 한 문장으로 설명해주세요"
)

print(response)


content='# RAG (Retrieval-Augmented Generation)\n\n**RAG는 대규모 언어모델이 외부 문서나 데이터베이스에서 관련 정보를 검색한 후, 그 정보를 바탕으로 더 정확하고 최신의 답변을 생성하는 기술입니다.**' additional_kwargs={} response_metadata={'id': 'msg_011CfHwYWHij9bzZQX43rQ5e', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 30, 'output_tokens': 100, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'} id='lc_run--01a0c7a0-faeb-7502-a4e2-1542f8ea1624-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 30, 'output_tokens': 100, 'total_tokens': 130, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 

In [3]:
llm.invoke(
    "RAG 이/가 무엇인지 한 문장으로 설명해주세요."
)

# Embedding 무엇인지 설명해주세요.
# Vector db 무엇인지 설명해주세요.
# -> 공통된 부분을 {변수}로 치환하여 템플릿처럼 prompt를 관리할 수 있다.

AIMessage(content='# RAG(Retrieval-Augmented Generation)\n\n**외부 데이터베이스에서 관련 정보를 검색하여 대형언어모델의 응답 생성에 활용하는 기술입니다.**', additional_kwargs={}, response_metadata={'id': 'msg_011CfHwYnGtS8kUkcjFvMt96', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 34, 'output_tokens': 68, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--01a0c7a1-0b57-71b1-b17e-01d6fa0f5713-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 34, 'output_tokens': 68, 'total_tokens': 102, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# prompt를 만든다.
prompt = ChatPromptTemplate.from_template(
    "{topic}에 대해 비전공자도 이해할 수 있도록 쉽게 설명해주세요."
)

# llm을 만든다.
llm= ChatAnthropic(
    model="claude-haiku-4-5",
    max_tokens=500,
)
# 3. AIMessage에서 문자열 추출을 위한 parser
parser = StrOutputParser()

# 4. promt와 llm을 연결한다.
chain = prompt | llm | parser

# 5. chanin을 실행한다.
response = chain.invoke({
    "topic": "RAG"
})

# 6. claude의 답변을 출력한다.
print(response)
print(type(response))


# RAG란? 쉬운 설명

## 핵심 개념: "책을 찾아본 후 답변하기"

RAG(Retrieval-Augmented Generation)는 **AI가 답변할 때 자료를 먼저 찾아본 후 그걸 바탕으로 대답하는 방식**입니다.

---

## 비유로 이해하기

### 일반 AI의 방식 (RAG 없음)
```
사용자: "2024년 애플 실적이 뭐야?"
AI: "제 학습 데이터는 2024년까지인데... 
    (뇌에만 의존해서 답변)
    대략 이 정도였던 것 같아요..."
❌ 최신 정보 없음, 잘못될 수 있음
```

### RAG 방식
```
사용자: "2024년 애플 실적이 뭐야?"
AI: "잠깐, 자료 찾아볼게요"
   → 📚 회사 보고서, 뉴스 검색
   → 찾은 자료를 읽음
   → "애플의 2024년 실적은... (정확한 자료 기반)"
✅ 최신 정보, 신뢰할 수 있음
```

---

## 실생활 예시

| 상황 | 일반 방식 | RAG 방식 |
|------|---------|--------|
| **학생이 숙제** | 머릿속으로만 생각 | 교과서 찾아보고 답변 |
| **고객 질문** | 직원의 기억에만 의존 | 회사 매뉴얼 확인하
<class 'langchain_core.messages.base.TextAccessor'>
